# Module 7: Output Generation (Briefing + Presentation)

This notebook verifies the final output generation module, compiling the validated Q&As, performance KPI tables, data gaps, and sentiment trend topics into an executive briefing document and a 16:9 widescreen dark-navy themed slide deck.

In [ ]:
# Cell 1: Initialization
import sys
from loguru import logger

logger.remove()
logger.add(sys.stderr, level="INFO")

from extraction_agent import EmbeddingService
from predictive_analyst_agent import PredictiveAnalystAgent
from answer_agent import AnswerAgent
from validation_agent import ValidationAgent
from output_generator import OutputOrchestrator

embedding_service = EmbeddingService()
analyst_agent = PredictiveAnalystAgent(embedding_service)
answer_agent = AnswerAgent()
validation_agent = ValidationAgent()
orchestrator = OutputOrchestrator()
print("All generating components initialized successfully.")

In [ ]:
# Cell 2: Run Analysis Pipeline
print("Analyzing Reliance Industries...")
analysis_results = await analyst_agent.run("Reliance Industries")
answerable = analysis_results["answerable"]
gaps = analysis_results["data_gaps"]
kpi_delta = analysis_results["kpi_delta"]
topics = analysis_results["topics"]

print("Answering predicted questions...")
qa_pairs = await answer_agent.run_batch(answerable[:5])

print("Validating responses...")
validation_output = await validation_agent.validate_batch(qa_pairs, kpi_delta, gaps)
print("Data readiness check complete.")

In [ ]:
# Cell 3: Compile Slides and Cheat Sheet
print("Compiling outputs...")
output_paths = orchestrator.run(
    company="Reliance Industries",
    quarter="Q4FY26",
    validation_output=validation_output,
    topics=topics,
    kpi_delta=kpi_delta
)
print("Outputs generated successfully!")

In [ ]:
# Cell 4: Verify Generated Brief Word Count
brief_path = output_paths["cheat_sheet_path"]
with open(brief_path, "r", encoding="utf-8") as f:
    brief_content = f.read()
    
print("=== CHEAT SHEET WORD COUNT & PREVIEW ===")
print(f"File Path: {brief_path}")
print(f"Word Count: {len(brief_content.split())}")
print("\n--- FIRST 600 CHARACTERS OF CHEAT SHEET ---")
print(brief_content[:600] + "...")

In [ ]:
# Cell 5: Verify PowerPoint Slide Count
from pptx import Presentation
pptx_path = output_paths["pptx_path"]
prs = Presentation(pptx_path)

print("=== PRESENTATION SLIDE COUNT & STRUCTURE ===")
print(f"File Path: {pptx_path}")
print(f"Total Slide Count: {len(prs.slides)}")
for idx, slide in enumerate(prs.slides):
    # Attempt to extract title/shapes
    slide_title = "[No title found]"
    # Find the title box or first text shape
    for shape in slide.shapes:
        if shape.has_text_frame and shape.text_frame.text.strip():
            lines = shape.text_frame.text.split('\n')
            slide_title = lines[0]
            break
    print(f"- Slide {idx+1}: {slide_title}")